# Part A Question 4: Bigram Sentence Probabilities

This notebook computes the target sentence probability using unsmoothed and add-one smoothed bigram language models.

In [28]:
import re
from collections import Counter
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import pandas as pd

In [29]:
DATA_PATH = Path('data/Data_3.txt')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/Data_3.txt')

text = DATA_PATH.read_text(encoding='utf-8')
print(text)


Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a different book by Danielle </s>



## 1. Pre-processing

In [30]:
def extract_padded_sentences(text: str) -> list[list[str]]:
    """Extract all <s> ... </s> sentences from the question text."""

    matches = re.findall(r'<s>\s+(.*?)\s+</s>', text)
    return [['<s>', *match.split(), '</s>'] for match in matches]


sentences = extract_padded_sentences(text)
training_sentences = sentences[:-1]
target_sentence = sentences[-1]

sentence_table = pd.DataFrame(
    {
        'Sentence Type': ['Training 1', 'Training 2', 'Training 3', 'Target'],
        'Tokens': [' '.join(sentence) for sentence in [*training_sentences, target_sentence]],
        'Token Count': [len(sentence) for sentence in [*training_sentences, target_sentence]],
    }
)

sentence_table


,Sentence Type,Tokens,Token Count
0,Training 1,<s> He read a book </s>,6
1,Training 2,<s> I read a different book </s>,7
2,Training 3,<s> He read a book by Danielle </s>,8
3,Target,<s> I read a different book by Danielle </s>,9


## 2. Counts and Preprocessed Data Visualization

In [31]:
def bigrams(sentence: list[str]) -> list[tuple[str, str]]:
    """Return adjacent token pairs for a padded sentence."""

    return list(zip(sentence, sentence[1:]))


unigram_counts = Counter(token for sentence in training_sentences for token in sentence)
bigram_counts = Counter(pair for sentence in training_sentences for pair in bigrams(sentence))
vocabulary = sorted(unigram_counts)
vocabulary_size = len(vocabulary)

vocabulary_table = pd.DataFrame({'Vocabulary Token': vocabulary})
unigram_count_table = pd.DataFrame(
    sorted(unigram_counts.items()),
    columns=['Token', 'Unigram Count'],
)
bigram_count_table = pd.DataFrame(
    [(previous, current, count) for (previous, current), count in sorted(bigram_counts.items())],
    columns=['Previous Token', 'Current Token', 'Bigram Count'],
)

print('Vocabulary size:', vocabulary_size)
print('Vocabulary:')
display(vocabulary_table)
print('Unigram counts:')
display(unigram_count_table)
print('Bigram counts:')
display(bigram_count_table)


Vocabulary size: 10
Vocabulary:


,Vocabulary Token
0,</s>
1,<s>
2,Danielle
3,He
4,I
5,a
6,book
7,by
8,different
9,read


Unigram counts:


,Token,Unigram Count
0,</s>,3
1,<s>,3
2,Danielle,1
3,He,2
4,I,1
5,a,3
6,book,3
7,by,1
8,different,1
9,read,3


Bigram counts:


,Previous Token,Current Token,Bigram Count
0,<s>,He,2
1,<s>,I,1
2,Danielle,</s>,1
3,He,read,2
4,I,read,1
5,a,book,2
6,a,different,1
7,book,</s>,2
8,book,by,1
9,by,Danielle,1


## 3. Unsmoothed Bigram Probability

In [32]:
@dataclass(frozen=True)
class BigramStep:
    """One conditional probability used in a sentence calculation."""

    bigram: tuple[str, str]
    numerator: int
    denominator: int
    probability: Fraction


def unsmoothed_bigram_step(bigram: tuple[str, str]) -> BigramStep:
    """Calculate P(w_i | w_i-1) without smoothing for one bigram."""

    previous_token, current_token = bigram
    numerator = bigram_counts[(previous_token, current_token)]
    denominator = unigram_counts[previous_token]

    return BigramStep(
        bigram=bigram,
        numerator=numerator,
        denominator=denominator,
        probability=Fraction(numerator, denominator),
    )


def sentence_probability(steps: list[BigramStep]) -> Fraction:
    """Multiply the probabilities for all bigrams in a sentence."""

    probability = Fraction(1, 1)
    for step in steps:
        probability *= step.probability
    return probability


target_bigrams = bigrams(target_sentence)
unsmoothed_steps = [unsmoothed_bigram_step(bigram) for bigram in target_bigrams]
unsmoothed_probability = sentence_probability(unsmoothed_steps)

unsmoothed_table = pd.DataFrame(
    {
        'Bigram': [f'{a} {b}' for a, b in [step.bigram for step in unsmoothed_steps]],
        'Count(wᵢ₋₁, wᵢ)': [step.numerator for step in unsmoothed_steps],
        'Count(wᵢ₋₁)': [step.denominator for step in unsmoothed_steps],
        'Probability': [f'{step.probability.numerator}/{step.probability.denominator}' for step in unsmoothed_steps],
    }
)

print('Unsmoothed sentence probability:', unsmoothed_probability, float(unsmoothed_probability))
unsmoothed_table


Unsmoothed sentence probability: 1/27 0.037037037037037035


,Bigram,"Count(wᵢ₋₁, wᵢ)",Count(wᵢ₋₁),Probability
0,<s> I,1,3,1/3
1,I read,1,1,1/1
2,read a,3,3,1/1
3,a different,1,3,1/3
4,different book,1,1,1/1
5,book by,1,3,1/3
6,by Danielle,1,1,1/1
7,Danielle </s>,1,1,1/1


## 4. Add-One Smoothed Bigram Probability

In [33]:
def smoothed_bigram_step(bigram: tuple[str, str]) -> BigramStep:
    """Calculate add-one smoothed P(w_i | w_i-1) for one bigram."""

    previous_token, current_token = bigram
    numerator = bigram_counts[(previous_token, current_token)] + 1
    denominator = unigram_counts[previous_token] + vocabulary_size

    return BigramStep(
        bigram=bigram,
        numerator=numerator,
        denominator=denominator,
        probability=Fraction(numerator, denominator),
    )


smoothed_steps = [smoothed_bigram_step(bigram) for bigram in target_bigrams]
smoothed_probability = sentence_probability(smoothed_steps)

smoothed_table = pd.DataFrame(
    {
        'Bigram': [f'{a} {b}' for a, b in [step.bigram for step in smoothed_steps]],
        'Count(wᵢ₋₁, wᵢ) + 1': [step.numerator for step in smoothed_steps],
        'Count(wᵢ₋₁) + V': [step.denominator for step in smoothed_steps],
        'Probability': [f'{step.probability.numerator}/{step.probability.denominator}' for step in smoothed_steps],
    }
)

print('Smoothed sentence probability:', smoothed_probability, float(smoothed_probability))
smoothed_table


Smoothed sentence probability: 512/418161601 1.2244070205767171e-06


,Bigram,"Count(wᵢ₋₁, wᵢ) + 1",Count(wᵢ₋₁) + V,Probability
0,<s> I,2,13,2/13
1,I read,2,11,2/11
2,read a,4,13,4/13
3,a different,2,13,2/13
4,different book,2,11,2/11
5,book by,2,13,2/13
6,by Danielle,2,11,2/11
7,Danielle </s>,2,11,2/11
